<a href="https://colab.research.google.com/github/cbonnin88/LimeLight/blob/main/People_Analytics_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from google.cloud import bigquery
from google.colab import auth

# **Authenticate**

In [3]:
auth.authenticate_user()
project_id = 'limelight-hr-data'
client = bigquery.Client(project=project_id)

# **Query the table that I build in dbt**

In [4]:
query='''
  SELECT * FROM `limelight-hr-data.dbt_limelight_analytics.dim_employee_directory`
'''
df_raw = client.query(query).to_dataframe()

# **Converting to a Polars DataFrame**

In [5]:
df_people = pl.from_pandas(df_raw)

In [6]:
display(df_people.head())

employee_id,first_name,last_name,gender,office_id,job_title,department,tenure,remote_type,performance,is_manager_flag,hours,country,base_salary,FTE_salary,salary_band,retention_risk_status
i64,str,str,str,i64,str,str,i64,str,str,bool,i64,str,f64,f64,str,str
10006,"""Joséphine""","""Bourgeois""","""Female""",1,"""ESG Strategy""","""ESG Strategy""",6,"""hybrid""","""average""",false,20,"""France""",49549.630841,91666.82,"""L3""","""Stable"""
10337,"""Nicolas""","""Gilson""","""Male""",7,"""ESG Strategy""","""ESG Strategy""",2,"""remote""","""bottom""",false,20,"""Belgium""",49549.630841,91666.82,"""L3""","""Stable"""
10781,"""Paula""","""Legros""","""Female""",7,"""ESG Strategy""","""ESG Strategy""",5,"""hybrid""","""bottom""",false,20,"""Belgium""",49549.630841,91666.82,"""L3""","""Stable"""
10802,"""Rodolfo""","""Jordá""","""Male""",3,"""ESG Strategy""","""ESG Strategy""",5,"""remote""","""bottom""",false,20,"""Spain""",49549.630841,91666.82,"""L3""","""Stable"""
10842,"""Robert""","""Marie""","""Male""",1,"""ESG Strategy""","""ESG Strategy""",3,"""remote""","""average""",false,20,"""France""",49549.630841,91666.82,"""L3""","""Stable"""


# **Pay Gap Analysis**
- Calculating the average salary by **Country, Department, and Gender** to see where the biggtrest discrepancies lie.

In [9]:
parity_analysis = (
    df_people.group_by(['country','department','gender'])
    .agg([
        pl.col('FTE_salary').mean().alias('avg_salary'),
        pl.len().alias('headcount')
    ])
    .sort(['country','department'])
)

display(parity_analysis.head())

country,department,gender,avg_salary,headcount
str,str,str,f64,u32
"""Austria""","""ESG Strategy""","""Non-Binary""",64760.2275,8
"""Austria""","""ESG Strategy""","""Female""",65243.156354,96
"""Austria""","""ESG Strategy""","""Male""",65442.475146,103
"""Austria""","""Engineering""","""Female""",95883.469855,69
"""Austria""","""Engineering""","""Non-Binary""",99007.634444,9


In [18]:
fig_parity = px.sunburst(
    parity_analysis.to_pandas(), # Plotly works better with pandas-like input
    path=['country','department','gender'],
    values = 'headcount',
    color ='avg_salary',
    color_continuous_scale = 'viridis_r',
    title = 'Workforce Distribution & Salary Heatmap'
)

fig_parity.show()

In [19]:
product_df = df_people.filter(pl.col('department') =='Product')

In [22]:
fig_box = px.box(
    product_df.to_pandas(),
    x='country',
    y='FTE_salary',
    color='gender',
    points='all', # Shows indivdual employees as dots
    title=' Product Salary Distribution by Country & Gender',
    notched=True # Shows if the medians differ significantly
)

fig_box.show()

In [25]:
fig_scatter = px.scatter(
    df_people.to_pandas(),
    x='FTE_salary',
    y='tenure',
    color='performance',
    symbol= 'retention_risk_status',
    hover_data=['first_name','last_name','job_title'],
    title='Tenure vs Salary (Colored by Performance)',
    labels={'FTE_salary':'Full-Time Salary Equivalent','tenure':'Tenure','performance':'Performance','retention_risk_status':'Retention Risk Status'}
)

fig_scatter.show()

In [27]:
band_summary = (
    df_people.group_by('salary_band')
    .agg([
        pl.col('FTE_salary').median().alias('median_salary'),
        pl.col('FTE_salary').min().alias('min_salary'),
        pl.col('FTE_salary').max().alias('max_salary'),
        pl.len().alias('emp_count')
    ])
    .sort('salary_band')
)
display(band_summary)

salary_band,median_salary,min_salary,max_salary,emp_count
str,f64,f64,f64,u32
"""L1""",230493.53,180657.09,334215.62,1536
"""L2""",119159.76,102136.94,142991.71,784
"""L3""",87301.73,81709.55,98614.97,2200
"""L4""",70512.94,61111.21,79710.28,4256
"""L5""",53921.66,49549.63,59139.88,3724


In [29]:
fig_joy = px.violin(
    df_people.to_pandas(),
    y='FTE_salary',
    x='salary_band',
    color='salary_band',
    box=True,
    points='outliers',
    hover_data=['first_name','last_name','department'],
    title='Salary Density & Banding Integrity Analysis',
    category_orders ={'salary_band':['L5','L4','L3','L2','L1']}
)

fig_joy.update_layout(
    xaxis_title = 'Assigned Salary Band',
    yaxis_title = 'FTE Salary (EUR)',
    showlegend= False
)